# Eksperimen 05: Stacking Ensemble (Fase M1 & M2)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(42)

print('Memuat data...')
train_df = pd.read_parquet('../../dataset/cleaned_dataset.parquet')
print(f'Ukuran awal: {train_df.shape}')


## M1: Post-Preprocessing EDA


In [ ]:
dups = train_df.duplicated(subset=['movie_title', 'cinema_ids', 'date_show']).sum()
print(f'Duplikasi: {dups}')
if dups > 0:
    print('Menghapus duplikasi...')
    train_df.drop_duplicates(subset=['movie_title', 'cinema_ids', 'date_show'], keep='first', inplace=True)

if not pd.api.types.is_datetime64_any_dtype(train_df['date_show']):
    train_df['date_show'] = pd.to_datetime(train_df['date_show'])
print('Tahap M1 Selesai')


## M1: Visualisasi Distribusi Tiket Awal
Tujuan: Memastikan tidak ada agregasi ganda yang lolos sebelum proses Sliding Window.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train_df['total_ticket'], bins=50, kde=True, color='teal')
plt.title('Distribusi Total Tiket per Hari (Sebelum Sliding Window)')
plt.xlim(0, train_df['total_ticket'].quantile(0.99))
plt.xlabel('Total Ticket')
plt.ylabel('Frekuensi')
plt.show()


## M2: Rekayasa Fitur Hibrida (Cuaca, Sineas, Kanibalisme)


In [ ]:
print('Mengekstrak fitur...')
if 'total_show' in train_df.columns:
    train_df.drop(columns=['total_show'], inplace=True)

# Proteksi jika cell di-run berkali-kali
train_df.drop(columns=['rainfall_mm', 'is_heavy_rain', 'weather_penalty_score', 'rain_intensity_impact', 'is_squeezed', 'inferred_release_date', 'day_since_release'], inplace=True, errors='ignore')

weather_df = pd.read_csv('../../dataset/weather_data.csv')
weather_df['date_show'] = pd.to_datetime(weather_df['date_show'])
train_df = train_df.merge(weather_df, on=['city_name', 'date_show'], how='left')
train_df['rainfall_mm'] = train_df['rainfall_mm'].fillna(0)
train_df['is_heavy_rain'] = train_df['is_heavy_rain'].fillna(0)

train_df['weather_penalty_score'] = train_df['is_heavy_rain'] * train_df['is_weekend']
train_df['rain_intensity_impact'] = train_df['is_heavy_rain'] * train_df['rainfall_mm']
train_df['is_squeezed'] = ((train_df['screen_share_pct'] < 15) & (train_df['is_clashing_blockbuster'] == 1)).astype('int8')

if 'release_date' in train_df.columns:
    train_df['release_date'] = pd.to_datetime(train_df['release_date'])
    train_df['day_since_release'] = (train_df['date_show'] - train_df['release_date']).dt.days + 1
else:
    first_show = train_df.groupby('movie_title')['date_show'].min().reset_index()
    first_show.rename(columns={'date_show': 'inferred_release_date'}, inplace=True)
    train_df = train_df.merge(first_show, on='movie_title', how='left')
    train_df['day_since_release'] = (train_df['date_show'] - train_df['inferred_release_date']).dt.days + 1
print('Fitur berhasil ditambahkan')


## M2: Transformasi Jendela Waktu (D1-D3) & Pre-Stacking Prep


In [ ]:
print('Memulai perakitan jendela waktu (ini mungkin butuh beberapa detik)...')
train_df.sort_values(by=['movie_title', 'cinema_ids', 'date_show'], inplace=True)
train_df['day_idx'] = train_df.groupby(['movie_title', 'cinema_ids']).cumcount() + 1

pair_counts = train_df.groupby(['movie_title', 'cinema_ids']).size()
valid_pairs = pair_counts[pair_counts >= 4].index
df_valid = train_df.set_index(['movie_title', 'cinema_ids']).loc[valid_pairs].reset_index()

history = df_valid[df_valid['day_idx'] <= 3]
history_agg = history.groupby(['movie_title', 'cinema_ids'], observed=True).agg(
    d1_ticket=('total_ticket', lambda x: x.iloc[0] if len(x)>0 else np.nan),
    d2_ticket=('total_ticket', lambda x: x.iloc[1] if len(x)>1 else np.nan),
    d3_ticket=('total_ticket', lambda x: x.iloc[2] if len(x)>2 else np.nan),
    scale_sp=('total_ticket', lambda x: max(x.mean(), 1))
).reset_index()

history_agg['D3_D1_ratio'] = history_agg['d3_ticket'] / history_agg['d1_ticket'].replace(0, 1)
p99 = history_agg['D3_D1_ratio'].quantile(0.99)
history_agg['D3_D1_ratio'] = np.clip(history_agg['D3_D1_ratio'], a_min=0, a_max=p99)

targets = df_valid[(df_valid['day_idx'] >= 4) & (df_valid['day_idx'] <= 10)]
train_final = targets.merge(history_agg, on=['movie_title', 'cinema_ids'], how='left')

for col in ['movie_title', 'cinema_ids', 'city_name', 'genre', 'age_rating']:
    if col in train_final.columns:
        train_final[col] = train_final[col].astype('category')
        
print(f'Dimensi data final (siap latih): {train_final.shape}')
display(train_final[['movie_title', 'day_since_release', 'total_ticket', 'scale_sp', 'D3_D1_ratio', 'weather_penalty_score']].head())


## M2: Visualisasi Efektivitas Fitur Hibrida (Ablation Pre-check)
Tujuan: Memverifikasi apakah cuaca ekstrim dan kanibalisme blockbuster benar-benar merusak penjualan tiket di hari tayang, serta menganalisis pola eksponensial peluruhan.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Efek Cuaca Buruk
sns.boxplot(x='weather_penalty_score', y='total_ticket', hue='weather_penalty_score', data=train_final, ax=axes[0], palette='Blues', legend=False)
axes[0].set_title('Efek Cuaca Akhir Pekan (Penalty) vs Tiket D4-D10')
axes[0].set_ylim(0, 500)
axes[0].set_xlabel('Weather Penalty (0=Aman, 1=Hujan Lebat)')

# 2. Efek Kanibalisme
sns.boxplot(x='is_squeezed', y='total_ticket', hue='is_squeezed', data=train_final, ax=axes[1], palette='Reds', legend=False)
axes[1].set_title('Kanibalisme Blockbuster vs Tiket D4-D10')
axes[1].set_ylim(0, 500)
axes[1].set_xlabel('Squeezed (0=Bebas, 1=Tergencet)')

# 3. Distribusi Rasio D3/D1
sns.histplot(train_final['D3_D1_ratio'].dropna(), bins=40, kde=True, ax=axes[2], color='orange')
axes[2].set_title('Distribusi Rasio Peluruhan (D3/D1 Ratio)')
axes[2].set_xlabel('D3/D1 Ratio (Clipped)')

plt.tight_layout()
plt.show()


## M3: Standardisasi Input Khusus Stacking
Tujuan: Resolusi mutlak *Missing Values* dan penyesuaian tipe `category` khusus untuk XGBoost dan LightGBM agar tidak meledak (*crash*).

In [ ]:
print('Memulai standardisasi input...')

# 1. Pisahkan tipe kolom berdasarkan deteksi otomatis
cat_candidates = train_final.select_dtypes(include=['object', 'category']).columns.tolist()
num_candidates = train_final.select_dtypes(include=['number']).columns.tolist()

# Pastikan ID dan target tidak ikut terolah secara keliru
target_cols = ['total_ticket', 'log_total_ticket', 'date_show', 'inferred_release_date']
num_candidates = [c for c in num_candidates if c not in target_cols]
cat_candidates = [c for c in cat_candidates if c not in target_cols]

# 2. Resolusi Kategorikal (Paksa menjadi 'category' pandas & isi NaNs)
for col in cat_candidates:
    if train_final[col].isnull().any():
        if train_final[col].dtype.name == 'category':
            if 'Unknown' not in train_final[col].cat.categories:
                train_final[col] = train_final[col].cat.add_categories(['Unknown'])
        train_final[col] = train_final[col].fillna('Unknown')
    train_final[col] = train_final[col].astype('category')

# 3. Resolusi Numerik (Gunakan Median untuk mengisi NaNs yang tersisa)
for col in num_candidates:
    if train_final[col].isnull().any():
        med_val = train_final[col].median()
        train_final[col] = train_final[col].fillna(med_val)

# 4. Final Check (Asersi Keamanan)
total_nans = train_final[cat_candidates + num_candidates].isnull().sum().max()
assert total_nans == 0, f'Masih terdapat {total_nans} Missing Value!'

print(f'Tahap M3 Selesai. Data Steril 100%.')
print(f'Daftar Fitur Kategori ({len(cat_candidates)}): {cat_candidates}')
print(f'Daftar Fitur Numerik ({len(num_candidates)}): {num_candidates}')


## M4: Post-FE EDA & Uji Keragaman
Tujuan: Verifikasi multikolinearitas (redundansi fitur numerik) dan uji homogenitas (variansi fitur kategori) sebelum pemodelan.

In [ ]:
print('--- UJI KORELASI KAUSAL (NUMERIK) ---')
plt.figure(figsize=(14, 10))
corr_matrix = train_final[num_candidates + ['log_total_ticket']].corr(method='pearson')
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Heatmap Korelasi Pearson: Fitur Numerik vs Target (log_total_ticket)')
plt.tight_layout()
plt.show()

print('\n--- UJI HOMOGENITAS (KATEGORIKAL) ---')
homogen_warnings = []
for col in cat_candidates:
    top_freq = train_final[col].value_counts(normalize=True).max()
    if top_freq > 0.99:
        homogen_warnings.append(f"{col} didominasi {top_freq*100:.2f}% oleh satu nilai tunggal.")

if homogen_warnings:
    print('PERINGATAN! Fitur berikut sangat homogen (Hampir tidak ada variansi / tidak berguna):')
    for w in homogen_warnings:
        print(' -', w)
else:
    print('Semua fitur kategorikal memiliki keragaman yang sehat (<99% dominasi nilai tunggal).')

print('\nTahap M4 Selesai. Matriks Input siap untuk masuk ke arena Stacking Ensemble (M5)!')


## M5: Pemodelan (Stacking Ensemble)
Tujuan: Melatih CatBoost, LGBM, XGBoost via GroupKFold dengan MASE Hacking, lalu di-blend menggunakan Ridge Regression.

In [ ]:
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

print('Mempersiapkan data dan model...')
X = train_final.drop(columns=['total_ticket', 'log_total_ticket', 'date_show', 'inferred_release_date', 'day_idx'])
y = train_final['total_ticket']
groups = train_final['movie_title']
weights = 1.0 / train_final['scale_sp'].replace(0, 1)

oof_cat = np.zeros(len(train_final))
oof_lgb = np.zeros(len(train_final))
oof_xgb = np.zeros(len(train_final))

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f'\n--- FOLD {fold+1}/5 ---')
    X_train, y_train, w_train = X.iloc[train_idx], y.iloc[train_idx], weights.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # 1. CATBOOST
    print('  > Training CatBoost...')
    model_cat = cb.CatBoostRegressor(iterations=300, loss_function='MAE', cat_features=cat_candidates, verbose=0, random_seed=42)
    model_cat.fit(X_train, y_train, sample_weight=w_train)
    oof_cat[val_idx] = np.clip(model_cat.predict(X_val), 0, None)

    # 2. LIGHTGBM
    print('  > Training LightGBM...')
    model_lgb = lgb.LGBMRegressor(n_estimators=300, objective='mae', random_state=42, n_jobs=-1, verbose=-1)
    model_lgb.fit(X_train, y_train, sample_weight=w_train)
    oof_lgb[val_idx] = np.clip(model_lgb.predict(X_val), 0, None)

    # 3. XGBOOST
    print('  > Training XGBoost...')
    model_xgb = xgb.XGBRegressor(n_estimators=300, objective='reg:absoluteerror', tree_method='hist', enable_categorical=True, random_state=42, n_jobs=-1)
    model_xgb.fit(X_train, y_train, sample_weight=w_train, verbose=False)
    oof_xgb[val_idx] = np.clip(model_xgb.predict(X_val), 0, None)

    gc.collect()

print('\n--- MENGGABUNGKAN MODEL DENGAN RIDGE REGRESSION ---')
X_meta = pd.DataFrame({'cat': oof_cat, 'lgb': oof_lgb, 'xgb': oof_xgb})
meta_model = Ridge(alpha=1.0, random_state=42, positive=True) # positive=True to prevent negative blending weights
meta_model.fit(X_meta, y)

train_final['oof_meta'] = np.clip(meta_model.predict(X_meta), 0, None)

print('Koefisien Kepercayaan Ridge Meta-Model:')
for name, coef in zip(X_meta.columns, meta_model.coef_):
    print(f' - {name.upper()}: {coef:.4f}')

mase_score = (np.abs(y - train_final['oof_meta']) / train_final['scale_sp'].replace(0, 1)).mean()
print(f'\n[FINAL OOF] Estimasi Stacking MASE Keseluruhan: {mase_score:.4f}')
